# Moteur de recherche d'articles scientifiques — Elements Impact

### Pipeline

#### Architecture du pipeline

```
Titre + Description du projet + Groupe ou Groupes bénéficiaires du projet + tags
            │
            ▼
┌─────────────────────────┐
│  ÉTAPE 1 — LLM          │  Sélection des prédicteurs pertinents
│  (Claude / GPT)         │  parmi les 79 du modèle Margolis
└──────────┬──────────────┘
           │  Liste de prédicteurs
           ▼
┌─────────────────────────┐
│  ÉTAPE 2 — OpenAlex     │  Recherche d'articles scientifiques
│  (API gratuite)         │  par prédicteur + contexte projet
└──────────┬──────────────┘
           │  Articles (titre, abstract, DOI)
           ▼
┌─────────────────────────┐
│  ÉTAPE 3 — LLM          │  Extraction de l'effect size
│  (extraction NLP)       │  Cohen's d / Hedge's g + durée
└──────────┬──────────────┘
           │
           ▼
   DataFrame structuré
   prêt pour Boussole

## Configurations

In [ ]:
# Configurations

import os, json, time, requests
import pandas as pd
from IPython.display import display, Markdown

# Clé API Anthropic
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

# Email pour obtenires requetes prioritaires
OPENALEX_EMAIL = "ksisaber@gmail.com"

# Fichier des prédicteurs Margolis utilisé par Elements
PREDICTORS_FILE = "articles_dataset.xlsx"

# Paramètres du moteur
TOP_N_PREDICTORS       = 8   # Nombre de prédicteurs à explorer par projet
ARTICLES_PER_PREDICTOR = 3   # Nombre d'articles à chercher par prédicteur
MIN_CITATIONS          = 3   # Filtre qualité : citations minimum par article

## Entrées du projet

In [2]:
# Inputs du projet

PROJECT = {

    # Nom court du projet
    "title": "Lieu d'accueil et de soutien à la parentalité — 1000 premiers jours",

    # Description des actions menées dans le projet
    "description": """
        Gestion d'un lieu d'accueil et de ressources dédié au soutien à la parentalité
        et à l'éveil des jeunes enfants durant leurs 1000 premiers jours.
        L'action vise à rompre l'isolement des familles à travers l'accompagnement
        professionnel et l'entraide entre pairs.
    """,

    # Groupe ou Groupes bénéficiaires du projet
    "target_group": "Les parents (en congé parental et actifs)",

    # Tags pour enrichir la recherche (optionnel non obligatoire)
    "tags": ["petite enfance", "parentalité", "lien social", "isolement"],

}

print(f"📋 Projet chargé : {PROJECT['title']}")
print(f"   Groupe cible  : {PROJECT['target_group']}")
print(f"   Tags          : {', '.join(PROJECT['tags'])}")

📋 Projet chargé : Lieu d'accueil et de soutien à la parentalité — 1000 premiers jours
   Groupe cible  : Les parents (en congé parental et actifs)
   Tags          : petite enfance, parentalité, lien social, isolement


## Moteur de recherche

In [3]:
# Moteur de recherche

# Chargement des prédicteurs

df_pred = pd.read_excel(PREDICTORS_FILE, sheet_name="Predictors")
df_pred = df_pred[["Predictor (EN)", "Predictor (FR)", "Domain (FR)"]].dropna(subset=["Predictor (EN)"])
df_pred.columns = ["predictor_en", "predictor_fr", "domain_fr"]

PREDICTORS_FOR_PROMPT = "\n".join(
    f"- [{row.domain_fr}] {row.predictor_en}"
    for _, row in df_pred.iterrows()
)


# Helpers

def call_claude(prompt: str, system: str = "", max_tokens: int = 2000) -> str:
    """Appelle Claude et retourne le texte de la réponse."""
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": max_tokens,
            "system": system,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60
    )
    resp.raise_for_status()
    return resp.json()["content"][0]["text"]


def parse_json(text: str):
    """Extrait et parse le JSON d'une réponse LLM."""
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]
    return json.loads(text.strip())


def reconstruct_abstract(idx: dict | None) -> str:
    """
    Reconstitue l'abstract depuis le format index inversé d'OpenAlex.
    OpenAlex stocke les abstracts sous la forme {mot: [positions]} pour
    des raisons de droit d'auteur. Cette fonction reconstitue le texte lisible.
    """
    if not idx:
        return ""
    slots = [""] * (max(p for pp in idx.values() for p in pp) + 1)
    for word, positions in idx.items():
        for pos in positions:
            slots[pos] = word
    return " ".join(slots)

### Étape 1 : Sélection des prédicteurs pertinents 

On utilise un LLM qui reçoit les 79 prédicteurs et sélectionne ceux que le projet est susceptible d'impacter avec un score de pertinence de 1 a 5

In [4]:
print("🔎 Étape 1/3 — Identification des prédicteurs pertinents...")

# Ce prompt guide le LLM à sélectionner les prédicteurs les plus pertinents pour le projet donné
resp_predictors = call_claude(
    system="Tu es expert en évaluation d'impact social et modèle Margolis SWB. Réponds UNIQUEMENT en JSON valide.",
    prompt=f"""
Projet : {PROJECT['title']}
Groupe : {PROJECT['target_group']}
Description : {PROJECT['description']}
Tags : {', '.join(PROJECT['tags'])}

Voici les 79 prédicteurs du modèle Margolis :
{PREDICTORS_FOR_PROMPT}

Sélectionne les {TOP_N_PREDICTORS} prédicteurs les plus susceptibles d'être impactés.
Pour chacun :
- predictor_en    : nom exact du prédicteur
- domain_fr       : domaine
- relevance_score : 1-5
- justification   : 1 phrase

JSON uniquement :
```json
[{{"predictor_en": "...", "domain_fr": "...", "relevance_score": 5, "justification": "..."}}]
```""",
    max_tokens=1500
)

selected = sorted(parse_json(resp_predictors), key=lambda x: x.get("relevance_score", 0), reverse=True)
print(f"   → {len(selected)} prédicteurs identifiés")

🔎 Étape 1/3 — Identification des prédicteurs pertinents...
   → 8 prédicteurs identifiés


### Étapes 2 & 3 : Recherche OpenAlex + Extraction effect size

Pour chaque prédicteur, on :
1. Construit une requête enrichie via LLM
2. Interroge OpenAlex
3. Extrait le Cohen's d / Hedge's g depuis les abstracts via LLM

In [5]:
results = []

for i, pred in enumerate(selected):
    p_name = pred["predictor_en"]
    print(f"\n📄 [{i+1}/{len(selected)}] {p_name}")

    # Construction de la requête — anglais uniquement, centré sur l'effect size
    fallback_query = f"{p_name} intervention effect size meta-analysis"
    try:
        query = call_claude(
            prompt=f"""Generate a short academic search query (4-6 words, ENGLISH ONLY) to find
meta-analyses or RCTs measuring the effect size of interventions on: {p_name}.
The query MUST include the words 'effect size' or 'meta-analysis'.
Respond with ONLY the query, no quotes, no punctuation, no explanation.""",
            max_tokens=30
        ).strip().strip('"').strip("'")
    except Exception:
        query = fallback_query
    print(f"   🔍 Requête : '{query}'")

    # Recherche OpenAlex
    oa_resp = requests.get(
        "https://api.openalex.org/works",
        params={
            "search"  : query,
            "per-page": ARTICLES_PER_PREDICTOR,
            "sort"    : "relevance_score:desc",
            "select"  : "title,abstract_inverted_index,doi,publication_year,cited_by_count,open_access,authorships",
            "mailto"  : OPENALEX_EMAIL,
        },
        timeout=30
    ).json().get("results", [])

    # Filtre qualité : citations minimum
    articles = [
        {
            "title"   : w.get("title", ""),
            "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
            "doi"     : w.get("doi", ""),
            "year"    : w.get("publication_year"),
            "cited"   : w.get("cited_by_count", 0),
            "oa"      : w.get("open_access", {}).get("is_oa", False),
        }
        for w in oa_resp
        if w.get("cited_by_count", 0) >= MIN_CITATIONS
    ]

    # Fallback si 0 résultats avec la requête LLM
    if not articles and query != fallback_query:
        print(f"   ⚠️ 0 résultats → requête de secours : '{fallback_query}'")
        oa_resp2 = requests.get(
            "https://api.openalex.org/works",
            params={
                "search"  : fallback_query,
                "per-page": ARTICLES_PER_PREDICTOR,
                "sort"    : "relevance_score:desc",
                "select"  : "title,abstract_inverted_index,doi,publication_year,cited_by_count,open_access,authorships",
                "mailto"  : OPENALEX_EMAIL,
            },
            timeout=30
        ).json().get("results", [])
        articles = [
            {
                "title"   : w.get("title", ""),
                "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
                "doi"     : w.get("doi", ""),
                "year"    : w.get("publication_year"),
                "cited"   : w.get("cited_by_count", 0),
                "oa"      : w.get("open_access", {}).get("is_oa", False),
            }
            for w in oa_resp2
            if w.get("cited_by_count", 0) >= MIN_CITATIONS
        ]

    print(f"   📰 {len(articles)} articles retenus (≥{MIN_CITATIONS} citations)")

    # Extraction des effect sizes
    for art in articles:
        abstract = art["abstract"][:2000] if art["abstract"] else "[Non disponible]"

        ext = call_claude(
            system="Tu es expert en méta-analyse. Extrais les tailles d'effet des abstracts. Réponds UNIQUEMENT en JSON valide.",
            prompt=f"""Article : {art['title']} ({art['year']})
Abstract : {abstract}

Prédicteur ciblé : {p_name}
Projet : {PROJECT['title']} | Groupe : {PROJECT['target_group']}

Retiens l'effect size le plus directement lié à {p_name}.

```json
{{
  "effect_size"     : <float ou null>,
  "effect_type"     : <"Cohen's d"|"Hedge's g"|"SMD"|"autre"|null>,
  "effect_duration" : <durée pendant laquelle l'effet persiste après l'intervention, ex: "6 months", "1 year", "2 years", "short-term only" — null si non mentionné>,
  "effect_direction" : <"positif"|"négatif"|null>,
  "relevance"       : <1-5>,
  "confidence"      : <1-5>,
  "source_text"     : <citation exacte du passage de l'abstract d'où l'effect size a été extrait, ou null si non trouvé>,
  "note"            : <string court>
}}
```""",
            max_tokens=500
        )

        try:
            parsed = parse_json(ext)
        except Exception:
            parsed = {}

        results.append({
            "predictor"      : p_name,
            "domain"         : pred["domain_fr"],
            "pred_relevance" : pred["relevance_score"],
            "pred_justif"    : pred["justification"],
            "search_query"   : query,
            "title"          : art["title"],
            "year"           : art["year"],
            "doi"            : art["doi"],
            "cited_by"       : art["cited"],
            "open_access"    : art["oa"],
            "effect_size"    : parsed.get("effect_size"),
            "effect_type"    : parsed.get("effect_type"),
            "effect_direction": parsed.get("effect_direction"),
            "effect_duration": parsed.get("effect_duration"),
            "art_relevance"  : parsed.get("relevance"),
            "confidence"     : parsed.get("confidence"),
            "source_text"    : parsed.get("source_text", ""),
            "note"           : parsed.get("note", ""),
        })
        time.sleep(0.3)

print("\n✅ Recherche terminée")


📄 [1/8] Social integration
   🔍 Requête : 'social integration interventions meta-analysis effect size'
   📰 3 articles retenus (≥3 citations)

📄 [2/8] Family support
   🔍 Requête : 'family support interventions meta-analysis effect'
   📰 3 articles retenus (≥3 citations)

📄 [3/8] Family education
   🔍 Requête : 'family education intervention effect size'
   📰 3 articles retenus (≥3 citations)

📄 [4/8] Social closeness
   🔍 Requête : 'meta-analysis social closeness intervention effects'
   📰 3 articles retenus (≥3 citations)

📄 [5/8] Positive relationships
   🔍 Requête : 'meta-analysis positive relationships interventions effect'
   📰 3 articles retenus (≥3 citations)

📄 [6/8] Access local services
   🔍 Requête : 'meta-analysis local services access interventions'
   📰 3 articles retenus (≥3 citations)

📄 [7/8] Stress management
   🔍 Requête : 'stress management interventions meta-analysis effect'
   📰 3 articles retenus (≥3 citations)

📄 [8/8] Self-efficacy
   🔍 Requête : 'meta-analys

## Résultats

In [6]:
# Résultats

df = pd.DataFrame(results)

# ── Résumé ───────────────────────────────────────────────────────────────────
n_total       = len(df)
n_with_effect = df["effect_size"].notna().sum()
n_predictors  = df["predictor"].nunique()
top_articles  = df[df["effect_size"].notna()].sort_values(
    ["art_relevance", "confidence", "cited_by"], ascending=False
)

display(Markdown(f"""
## 📊 Résultats — {PROJECT['title']}
**Groupe cible :** {PROJECT['target_group']}

| Métrique | Valeur |
|---|---|
| Prédicteurs explorés | {n_predictors} |
| Articles analysés | {n_total} |
| Articles avec effect size | {n_with_effect} ({n_with_effect/n_total:.0%} si >0) |
"""))

# ── Prédicteurs sélectionnés ─────────────────────────────────────────────────
display(Markdown("### Prédicteurs identifiés"))
display(
    pd.DataFrame(selected)[["predictor_en", "domain_fr", "relevance_score", "justification"]]
    .rename(columns={
        "predictor_en"  : "Prédicteur",
        "domain_fr"     : "Domaine",
        "relevance_score": "Score",
        "justification" : "Justification"
    })
)

# ── Articles avec effect size ─────────────────────────────────────────────────
display(Markdown("### 📄 Articles avec effect size extrait"))

if not top_articles.empty:
    display(
        top_articles[[
            "predictor", "title", "year", "cited_by",
            "effect_size", "effect_type", "effect_duration",
            "art_relevance", "confidence", "note", "doi"
        ]].rename(columns={
            "predictor"      : "Prédicteur",
            "title"          : "Titre",
            "year"           : "Année",
            "cited_by"       : "Citations",
            "effect_size"    : "Effect size",
            "effect_type"    : "Type",
            "effect_duration": "Durée",
            "art_relevance"  : "Pertinence",
            "confidence"     : "Confiance",
            "note"           : "Note",
            "doi"            : "DOI",
        })
    )
else:
    print("Aucun article avec effect size extrait. Essayez de réduire MIN_CITATIONS.")

# ── Export Excel ──────────────────────────────────────────────────────────────
output_file = "resultats_moteur.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.sort_values(["art_relevance", "confidence"], ascending=False).to_excel(
        writer, sheet_name="Tous les articles", index=False
    )
    top_articles.groupby("predictor").first().reset_index().to_excel(
        writer, sheet_name="Meilleur par prédicteur", index=False
    )

print(f"\n💾 Résultats exportés → {output_file}")


## 📊 Résultats — Lieu d'accueil et de soutien à la parentalité — 1000 premiers jours
**Groupe cible :** Les parents (en congé parental et actifs)

| Métrique | Valeur |
|---|---|
| Prédicteurs explorés | 8 |
| Articles analysés | 24 |
| Articles avec effect size | 6 (25% si >0) |


### Prédicteurs identifiés

,Prédicteur,Domaine,Score,Justification
0,Social integration,Extra-personnel,5,Le lieu d'accueil favorise directement l'intég...
1,Family support,Extra-personnel,5,L'accompagnement professionnel et l'entraide e...
2,Family education,Environnement matériel & éducation,5,Le soutien à la parentalité apporte des ressou...
3,Social closeness,Extra-personnel,4,Les interactions régulières au sein du lieu d'...
4,Positive relationships,Extra-personnel,4,L'environnement bienveillant et les activités ...
5,Access local services,Environnement matériel & éducation,4,Le lieu constitue un point d'accès facilité au...
6,Stress management,Intra-personnel,4,L'accompagnement professionnel et le soutien p...
7,Self-efficacy,Intra-personnel,4,Les ressources et l'accompagnement proposés re...


### 📄 Articles avec effect size extrait

,Prédicteur,Titre,Année,Citations,Effect size,Type,Durée,Pertinence,Confiance,Note,DOI
18,Stress management,Effects of occupational stress management inte...,2008,1357,0.526,Cohen's d,None,5.0,5.0,Effet global des interventions de gestion du s...,https://doi.org/10.1037/1076-8998.13.1.69
23,Self-efficacy,What are the most effective intervention techn...,2011,705,0.160,Cohen's d,None,5.0,5.0,Effect size spécifiquement pour le changement ...,https://doi.org/10.1093/her/cyr005
3,Family support,The Effect of Family Interventions on Relapse ...,2001,652,0.200,autre,first years after hospitalization,5.0,4.0,Réduction de 20% du taux de rechute grâce à l'...,https://doi.org/10.1093/oxfordjournals.schbul....
13,Positive relationships,The effectiveness of interprofessional educati...,2018,441,1.370,autre,None,4.0,4.0,IPE améliore les relations de travail collabor...,https://doi.org/10.1016/j.kjms.2017.12.009
1,Social integration,Effects of parenting education with expectant ...,2010,172,0.300,Cohen's d,None,4.0,4.0,Le développement social de l'enfant est l'indi...,https://doi.org/10.1037/a0019691
21,Self-efficacy,A Meta‐Analysis of Research on Protection Moti...,2000,2433,0.520,Cohen's d,None,4.0,3.0,Taille d'effet globale incluant self-efficacy ...,https://doi.org/10.1111/j.1559-1816.2000.tb023...



💾 Résultats exportés → resultats_moteur.xlsx


## EXPORT FORMAT BOUSSOLE

In [7]:
# Mapping domaines FR → EN (format attendu par Boussole)
DOMAIN_FR_TO_EN = {
    'Environnement socio-politique':      'Socio-political environment',
    'Environnement matériel & éducation': 'Material environment and education',
    'Extra-personnel':                    'Extra-personal',
    'Intra-personnel':                    'Intra-personal',
    'Santé':                              'Health',
    'Travail & Activités':                'Work and Activities',
}

def make_boussole_df(df_src, project):
    """
    Format Boussole pour un sous-ensemble de df (avec ou sans effect size).
    Triés par prédicteur puis pertinence/confiance décroissante.

    Note (raw) :
      1. Passage exact de l'abstract d'où l'effect size a été extrait
      2. Note complémentaire du LLM
      3. Référence de l'article (titre, année, citations)
      4. Détails techniques (type, direction, confiance)
    """
    df_sorted = df_src.sort_values(
        ['predictor', 'art_relevance', 'confidence'],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    tags_str = ', '.join(project.get('tags', []))
    rows = []
    for _, r in df_sorted.iterrows():
        domain_en = DOMAIN_FR_TO_EN.get(r.get('domain', ''), r.get('domain', ''))

        # ── Note (raw) — passage source de l'effect size ────────────────────
        note_parts = []

        # 1. Passage exact d'où l'effect size a été extrait
        if r.get('source_text'):
            note_parts.append(f'Extrait : "{r["source_text"]}"')

        # 2. Note complémentaire du LLM
        if r.get('note'):
            note_parts.append(r['note'])

        # 3. Référence article : titre + année + citations
        article_ref = r.get('title', '')
        if r.get('year'):     article_ref += f" ({r['year']})"
        if r.get('cited_by'): article_ref += f" — {r['cited_by']} citations"
        if article_ref:
            note_parts.append(f"Source : {article_ref}")

        # 4. Détails techniques
        details = []
        if r.get('effect_type'):      details.append(f"Type : {r['effect_type']}")
        if r.get('effect_direction'): details.append(f"Direction : {r['effect_direction']}")
        if r.get('confidence'):       details.append(f"Confiance LLM : {r['confidence']}/5")
        if details:
            note_parts.append(' | '.join(details))

        rows.append({
            'Project':             project.get('title', ''),
            'Project description': project.get('description', ''),
            'Tags':                tags_str,
            'Group':               project.get('target_group', ''),
            'Domain':              domain_en,
            'Predictor':           r['predictor'],
            'Effect size':         r['effect_size'],
            'Effect duration':     r.get('effect_duration', ''),
            'Note (raw)':          ' ; '.join(note_parts),
            'Article links':       r.get('doi', ''),
        })
    return pd.DataFrame(rows)

# Construction des deux DataFrames
boussole_with    = make_boussole_df(df[df['effect_size'].notna()], PROJECT)
boussole_without = make_boussole_df(df[df['effect_size'].isna()],  PROJECT)

display(Markdown(f'### 📋 Format Boussole — avec effect size ({len(boussole_with)} lignes)'))
display(boussole_with)

display(Markdown(f'### 📋 Format Boussole — sans effect size ({len(boussole_without)} lignes)'))
display(boussole_without)

output_file = 'resultats_moteur_boussole3.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Onglet 1 : articles avec effect size — prêt à injecter dans Boussole
    boussole_with.to_excel(writer, sheet_name='Boussole — avec effect size', index=False)
    # Onglet 2 : articles sans effect size — à compléter manuellement
    boussole_without.to_excel(writer, sheet_name='Boussole — sans effect size', index=False)
    # Onglet 3 : tous les articles avec colonnes techniques complètes
    df.sort_values(['art_relevance', 'confidence'], ascending=False).to_excel(
        writer, sheet_name='Tous les articles', index=False
    )

print(f'\n💾 Export : {output_file}')
print(f'   Onglet 1 : Boussole — avec effect size  ({len(boussole_with)} lignes)')
print(f'   Onglet 2 : Boussole — sans effect size  ({len(boussole_without)} lignes)')
print(f'   Onglet 3 : Tous les articles')
print(f'\n   {df["predictor"].nunique()} prédicteur(s) explorés au total')

### 📋 Format Boussole — avec effect size (6 lignes)

,Project,Project description,Tags,Group,Domain,Predictor,Effect size,Effect duration,Note (raw),Article links
0,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Family support,0.200,first years after hospitalization,"Extrait : ""The main result of the meta-analysi...",https://doi.org/10.1093/oxfordjournals.schbul....
1,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Positive relationships,1.370,None,"Extrait : ""The effect summary value of 1.37 wi...",https://doi.org/10.1016/j.kjms.2017.12.009
2,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Intra-personal,Self-efficacy,0.160,None,"Extrait : ""A small, yet significant (P < 0.01)...",https://doi.org/10.1093/her/cyr005
3,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Intra-personal,Self-efficacy,0.520,None,"Extrait : ""The mean overall effect size ( d +=...",https://doi.org/10.1111/j.1559-1816.2000.tb023...
4,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Social integration,0.300,None,"Extrait : ""social development (d = .30)"" ; Le ...",https://doi.org/10.1037/a0019691
5,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Intra-personal,Stress management,0.526,None,"Extrait : ""The overall weighted effect size (C...",https://doi.org/10.1037/1076-8998.13.1.69


### 📋 Format Boussole — sans effect size (18 lignes)

,Project,Project description,Tags,Group,Domain,Predictor,Effect size,Effect duration,Note (raw),Article links
0,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Access local services,NaN,None,L'abstract concerne uniquement des recommandat...,https://doi.org/10.1097/aln.0b013e31823c1030
1,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Access local services,NaN,None,Cet abstract présente des statistiques descrip...,https://doi.org/10.3322/caac.20107
2,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Access local services,NaN,None,Abstract non disponible - impossible d'extrair...,https://doi.org/10.1007/s11606-018-4818-7
3,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Family education,NaN,None,"Extrait : ""The effect sizes associated with mo...",https://doi.org/10.1093/heapro/dar075
4,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Family education,NaN,None,Cet abstract présente des données épidémiologi...,https://doi.org/10.1111/j.1467-789x.2004.00133.x
5,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Material environment and education,Family education,NaN,None,Abstract non disponible - impossible d'extrair...,https://doi.org/10.1186/s12916-018-1244-y
6,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Family support,NaN,None,"Extrait : ""Meta-analysis indicated that althou...",https://doi.org/10.3322/caac.20081
7,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Family support,NaN,None,Abstract non disponible - impossible d'extrair...,https://doi.org/10.1016/s0140-6736(11)61094-5
8,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Positive relationships,NaN,None,Aucune taille d'effet liée aux relations posit...,https://doi.org/10.1016/j.smrv.2021.101556
9,Lieu d'accueil et de soutien à la parentalité ...,\n Gestion d'un lieu d'accueil et de re...,"petite enfance, parentalité, lien social, isol...",Les parents (en congé parental et actifs),Extra-personal,Positive relationships,NaN,None,Cet abstract porte sur la pratique mentale et ...,https://doi.org/10.1037/0021-9010.79.4.481



💾 Export : resultats_moteur_boussole3.xlsx
   Onglet 1 : Boussole — avec effect size  (6 lignes)
   Onglet 2 : Boussole — sans effect size  (18 lignes)
   Onglet 3 : Tous les articles

   8 prédicteur(s) explorés au total
